# DocSage Evaluation & Failure Analysis

This notebook runs RAGAS over both the naive and tuned pipelines, builds the before/after table, and documents failure cases.

In [2]:
# ── Fix sys.path so 'docsage' package is importable from notebooks/ ──
import sys
from pathlib import Path

# This resolves to: <repo_root>/  (two levels up from notebooks/)
project_root = Path().resolve().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: C:\Users\SANGAI\Desktop\Source Cit. RAG


In [4]:
import json
import os
import pandas as pd

from docsage.src.evaluate import run_eval_for_collection

naive_scores = run_eval_for_collection("naive")
tuned_scores = run_eval_for_collection("tuned")

naive_faith     = naive_scores["faithfulness"][0]
tuned_faith     = tuned_scores["faithfulness"][0]
naive_ans_rel   = naive_scores["answer_relevancy"][0]
tuned_ans_rel   = tuned_scores["answer_relevancy"][0]
naive_ctx_prec  = naive_scores["context_precision"][0]
tuned_ctx_prec  = tuned_scores["context_precision"][0]
naive_ctx_rec   = naive_scores["context_recall"][0]
tuned_ctx_rec   = tuned_scores["context_recall"][0]

def pct_delta(a, b):
    if a == 0:
        return "n/a"
    return f"{(b - a) / a * 100:.1f}%"

halluc_naive     = 1 - naive_faith
halluc_tuned     = 1 - tuned_faith
halluc_reduction = (halluc_naive - halluc_tuned) / halluc_naive * 100 if halluc_naive != 0 else 0

table = pd.DataFrame(
    [
        ["Faithfulness",       naive_faith,        tuned_faith,        pct_delta(naive_faith, tuned_faith)],
        ["Answer Relevancy",   naive_ans_rel,      tuned_ans_rel,      pct_delta(naive_ans_rel, tuned_ans_rel)],
        ["Context Precision",  naive_ctx_prec,     tuned_ctx_prec,     pct_delta(naive_ctx_prec, tuned_ctx_prec)],
        ["Context Recall",     naive_ctx_rec,      tuned_ctx_rec,      pct_delta(naive_ctx_rec, tuned_ctx_rec)],
        ["Hallucination Rate*",halluc_naive * 100, halluc_tuned * 100, f"{halluc_reduction:.1f}%"],
    ],
    columns=["Metric", "Naive Chunking", "Tuned + Hybrid + Rerank", "Δ Change"],
)
table

ModuleNotFoundError: No module named 'datasets'

> *Hallucination Rate = 1 - faithfulness score, expressed as a percentage*

## Failure Case Analysis (Tuned Pipeline)

Fill this section after running the evaluation and inspecting low-faithfulness examples.

QUESTION: "<Paste the question that failed here>"

RETRIEVED CHUNK (wrong):

```text
<Paste the actual retrieved chunk text here>
```

SOURCE: <filename>, page <X>

WHY IT FAILED: <Your analysis of why retrieval/reranking selected the wrong chunk>

DETECTION METHOD: <e.g., "RAGAS faithfulness score dropped to 0.12 for this question, flagging it for manual review">

FIX APPLIED: <Describe the change you made to the pipeline or chunking>

RESULT AFTER FIX: <Describe how the metrics and/or this specific question's behavior improved>